# Lesson 3.3 - App Of Apps Wiring

**Level: Intermediate**

## Learning Objectives
- Build ArgoCD app-of-apps manifest.
- Point ArgoCD at `gitops/` in the same repository.
- Add Flux Kustomization that applies app-of-apps definitions.

**Project file(s) produced:** `bootstrap/apps/app-of-apps.yaml`, `bootstrap/apps/kustomization.yaml`, `bootstrap/argocd-apps-kustomization.yaml`

## Concepts
Flux installs ArgoCD, then ArgoCD takes over workload-level GitOps.
This handoff is implemented by app-of-apps and a Flux Kustomization that applies it.

In [ ]:
%%writefile bootstrap/apps/app-of-apps.yaml
apiVersion: argoproj.io/v1alpha1
kind: Application
metadata:
  name: app-of-apps
  namespace: argocd
  finalizers:
    - resources-finalizer.argocd.argoproj.io
spec:
  project: default
  source:
    repoURL: git@github.com:sl31pn1r/aionk8s.git
    targetRevision: master
    path: gitops
  destination:
    server: https://kubernetes.default.svc
    namespace: argocd
  syncPolicy:
    automated:
      prune: true
      selfHeal: true
    syncOptions:
      - CreateNamespace=true

In [ ]:
%%writefile bootstrap/apps/kustomization.yaml
apiVersion: kustomize.config.k8s.io/v1beta1
kind: Kustomization

resources:
  - app-of-apps.yaml

In [ ]:
%%writefile bootstrap/argocd-apps-kustomization.yaml
apiVersion: kustomize.toolkit.fluxcd.io/v1
kind: Kustomization
metadata:
  name: argocd-apps
  namespace: flux-system
spec:
  interval: 5m
  path: ./bootstrap/apps
  prune: true
  sourceRef:
    kind: GitRepository
    name: fc-cnandk8s-sbox-swec-01
  dependsOn:
    - name: fc-cnandk8s-sbox-swec-01-bootstrap
  timeout: 5m

## Exercise
Replace hardcoded repo/branch values with your repository details and list how you would store these safely for teams.

In [ ]:
# TODO: Update repoURL and targetRevision in app-of-apps.yaml
# TODO: Validate ArgoCD can read the repo using a deploy-key secret
print('Then run: kubectl apply -f bootstrap/argocd-apps-kustomization.yaml')

---
**Next:** [Lesson 4.1 - ApplicationSet And Namespace Scaffolding](../phase-4-gitops-ai-workloads/lesson-4.1-applicationset-and-namespace-scaffolding.ipynb)